In [ ]:
from langchain.llms import Ollama
import pyaudio
import asyncio
import websockets
import json
import pyttsx3
import nest_asyncio
import time

# Initialize Ollama model
ollama = Ollama(base_url='http://localhost:11434', model='llama3.2:1b')

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Apply nest_asyncio for asyncio to work in Jupyter
nest_asyncio.apply()

# Your Deepgram API key
# NOTE: the original key that lived here has been revoked/redacted before
# this project was made public. See services/config.py for the current,
# env-var-based approach used by the rebuilt app.
DEEPGRAM_API_KEY = "<REDACTED>"

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024

# Audio queue for processing audio data
audio_queue = asyncio.Queue()
stop_execution = False
is_muted = False  # Flag to handle microphone muting

# Initialize chat history for context
chat_history = [{"role": "system", "content": "You are a helpful assistant. Respond concisely and keep the conversation flowing."}]

# Global variable to track time since last audio input
last_input_time = time.time()

# Pause handling variables
PAUSE_THRESHOLD = 2.0  # Seconds to consider a pause in speech

# Callback function to handle audio data
def callback(input_data, frame_count, time_info, status_flags):
    global last_input_time, is_muted
    if not is_muted:
        audio_queue.put_nowait(input_data)
        last_input_time = time.time()  # Update the time with every audio input
    return (input_data, pyaudio.paContinue)

# Microphone function to capture live audio
async def microphone():
    global stop_execution
    audio = pyaudio.PyAudio()
    stream = audio.open(
        format=FORMAT,
        channels=CHANNELS,
        rate=RATE,
        input=True,
        frames_per_buffer=CHUNK,
        stream_callback=callback
    )
    stream.start_stream()

    try:
        while stream.is_active() and not stop_execution:
            await asyncio.sleep(0.1)
    finally:
        stream.stop_stream()
        stream.close()
        audio.terminate()

# Function to continuously send audio data
async def send_audio_data(ws):
    global stop_execution
    while not stop_execution:
        if not audio_queue.empty() and not is_muted:
            data = await audio_queue.get()
            await ws.send(data)
        await asyncio.sleep(0.1)

# Function to receive transcriptions and interact with Ollama
async def receiver_and_interact(ws):
    global chat_history, is_muted, last_input_time, stop_execution
    pending_transcript = ""

    while not stop_execution:
        try:
            msg = await ws.recv()
            msg = json.loads(msg)

            if 'channel' in msg:
                transcript = msg['channel']['alternatives'][0]['transcript']
                if transcript:
                    print(f"Partial Transcript: {transcript}")
                    pending_transcript += transcript + " "
                    last_input_time = time.time()

            # Check for pause and finalize transcript
            if time.time() - last_input_time:
                user_input = pending_transcript.strip()
                if user_input != "":
                    print(f"Final Input: {user_input}")

                if "bye" in user_input.lower():
                    print("Detected 'bye'. Stopping execution...")
                    stop_execution = True
                    break

                # Append user input to chat history
                chat_history.append({"role": "user", "content": user_input})

                # Mute microphone during TTS
                is_muted = True

                if user_input:
                    try:
                        print(f"Sending to Ollama: {user_input}")
                        response = ollama.invoke(f"{user_input}")
                        print(f"Ollama response: {response}")
                    except Exception as e:
                        print(f"Error invoking Ollama: {e}")
                        response = "Sorry, I couldn't process that input."

                    # Convert response to speech
                    engine.say(response)
                    engine.runAndWait()

                    # Append response to chat history
                    chat_history.append({"role": "assistant", "content": response})

                # Unmute microphone
                is_muted = False

                # Reset pending transcript
                pending_transcript = ""

        except websockets.exceptions.ConnectionClosedError:
            print("WebSocket connection closed unexpectedly: Retrying...")
            await asyncio.sleep(5)
        except Exception as e:
            print(f"Unexpected error in receiver: {e}")

# Function to process the audio and interact with Deepgram
async def process():
    global stop_execution
    extra_headers = {
        'Authorization': f'token {DEEPGRAM_API_KEY}'
    }

    wss_url = 'wss://api.deepgram.com/v1/listen?encoding=linear16&sample_rate=16000&channels=1&punctuate=true&model=general'

    reconnect_attempt = 0  # Track reconnection attempts

    while not stop_execution:
        try:
            async with websockets.connect(
                wss_url,
                extra_headers=extra_headers
            ) as ws:
                print("WebSocket connected")
                reconnect_attempt = 0  # Reset attempt counter on successful connection
                await asyncio.gather(send_audio_data(ws), receiver_and_interact(ws))

        except websockets.exceptions.ConnectionClosedError:
            print("WebSocket connection closed unexpectedly: Retrying...")

        # Apply exponential backoff for reconnection
        reconnect_attempt += 1
        backoff_time = min(2 ** reconnect_attempt, 30)  # Cap backoff time at 30 seconds
        print(f"Reconnecting in {backoff_time} seconds...")
        await asyncio.sleep(backoff_time)

# Main function to start the process
async def main():
    global stop_execution
    try:
        await asyncio.gather(microphone(), process())
    except asyncio.CancelledError:
        pass
    finally:
        print("Shutting down...")
        stop_execution = True

# Run the transcription process
if __name__ == "__main__":
    print("Running transcription and conversation. Say something to start.")
    asyncio.run(main())
